In [ ]:
# Pin BLAS/OpenMP to 1 thread BEFORE importing numpy/scipy/tenpy.
# Small TN tensors (DMRG/TDVP/PEPS) => multi-threaded BLAS oversubscribes
# cores and runs 10-40x SLOWER on a loaded box. This cell must stay first
# and run before any import: the libraries read these variables only once,
# at import time.
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")
print("BLAS/OpenMP threads pinned to", os.environ.get("OMP_NUM_THREADS"))

In [ ]:
import os
import time
from pathlib import Path
from itertools import product

import numpy as np
import matplotlib.pyplot as plt

os.environ.setdefault("CUPY_CACHE_DIR", str(Path.cwd() / ".cache" / "cupy"))
Path(os.environ["CUPY_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)

from ryd_gate import DEFAULT_C6, RydbergSystem, simulate
from ryd_gate.lattice import Register
from ryd_gate.core.level_structures import InteractionSpec
from ryd_gate.protocols.digital_analog import DigitalAnalogProtocol
from ryd_gate.backends.tenpy_mps import mps_fidelity, product_superposition_mps
from ryd_gate.backends.tn_common.compiler import tn_lattice_spec_from_system


In [ ]:
# Local compatibility shim: the kernel DigitalAnalogProtocol is now function-
# based (omega_R_fn(t), ...). These notebooks keep their piecewise-constant
# Segment schedules and lower them to step functions here. Note the backend
# integrates on n_steps uniform slices, so segment edges should be resolved
# by a generous n_steps (the notebooks already scale it with len(segments)).
from dataclasses import dataclass

import numpy as np


@dataclass
class Segment:
    duration: float
    omega_R: "float | np.ndarray" = 0.0
    omega_hf: "float | np.ndarray" = 0.0
    delta_R: "float | np.ndarray" = 0.0
    delta_hf: "float | np.ndarray" = 0.0


def segments_protocol(segments, n_steps=200):
    """Build the function-based DigitalAnalogProtocol from constant segments."""
    segments = list(segments)
    edges = np.cumsum([0.0] + [float(seg.duration) for seg in segments])

    def step_fn(name):
        values = [getattr(seg, name) for seg in segments]

        def fn(t):
            j = int(np.clip(np.searchsorted(edges, t, side="right") - 1, 0, len(values) - 1))
            return values[j]

        return fn

    protocol = DigitalAnalogProtocol(
        t_gate=float(edges[-1]),
        omega_R_fn=step_fn("omega_R"),
        omega_hf_fn=step_fn("omega_hf"),
        delta_R_fn=step_fn("delta_R"),
        delta_hf_fn=step_fn("delta_hf"),
        n_steps=n_steps,
    )
    protocol.segments = segments  # keep the schedule inspectable for plotting
    return protocol


# Plus-state preparation benchmark

The whole Hamiltonian is
$$
\begin{align*}
H(t)/\hbar =
&\sum_i \frac{\Omega_{R,i}(t)}{2}
\left(|r_i\rangle\langle 1_i|+\mathrm{h.c.}\right)
-\sum_i \Delta_{R,i}(t)n_i^r\
&+\sum_i \frac{\Omega_{\rm hf,i}(t)}{2}
\left(|1_i\rangle\langle0_i|+\mathrm{h.c.}\right)
-\sum_i \Delta_{\rm hf,i}(t)n_i^1
+\sum_{i<j}V_{ij}n_i^r n_j^r .
\end{align*}
$$

This notebook keeps the benchmark deliberately direct: each backend has its own executable block, with no `RUN_*` switches. All tensor-network comparisons use nearest-neighbor interactions (`mode="nn"`) so the TN algorithms see a local Hamiltonian and compare the same physics.

For this explicit `01r` three-level plus-preparation problem, the currently relevant TN paths are `mps` (TeNPy MPS-TDVP), `peps` (YASTN finite PEPS), and `gputn` (CUDA/cuQuantum experimental path).


In [ ]:
# Physical parameters and the two-stage pulse shape.
OMEGA_R = 2 * np.pi * 6e6
OMEGA_HF = 2 * np.pi * 6e6

t_pi_R = np.pi / OMEGA_R
t_pi2_R = np.pi / (2 * OMEGA_R)

DELTA_R_ISING = 2 * np.pi * 1.0e6
T_ISING = 1.0e-6

full_segments = [
    Segment(duration=t_pi2_R, omega_R=OMEGA_R),
    Segment(
        duration=T_ISING,
        omega_R=0.0,
        omega_hf=0.0,
        delta_R=DELTA_R_ISING,
        delta_hf=0.0,
    ),
]

t_prep_end = full_segments[0].duration
t_final = sum(seg.duration for seg in full_segments)
_proto_fields = (
    ("omega_R", r"$\Omega_R$"),
    ("omega_hf", r"$\Omega_{\rm hf}$"),
    ("delta_R", r"$\Delta_R$"),
    ("delta_hf", r"$\Delta_{\rm hf}$"),
)

_t_edges = [0.0]
for seg in full_segments:
    _t_edges.append(_t_edges[-1] + seg.duration)

fig_p, axes_p = plt.subplots(4, 1, figsize=(12, 7), sharex=True)
for ax_p, (field, label) in zip(axes_p, _proto_fields):
    t_pts, y_pts = [], []
    for j, seg in enumerate(full_segments):
        v = getattr(seg, field)
        t_pts.extend([_t_edges[j], _t_edges[j + 1]])
        y_pts.extend([v, v])
    ax_p.step(t_pts, y_pts, where="post", lw=2, color="tab:blue")
    ax_p.axhline(0.0, color="k", ls=":", lw=0.8)
    ax_p.axvline(t_prep_end, color="tab:orange", ls="--", lw=1, alpha=0.8)
    ax_p.set_ylabel(label)
    ax_p.grid(alpha=0.3)
    ax_p.set_xlim(0.0, t_final)

axes_p[-1].set_xlabel("time (s)")
axes_p[-1].axvline(t_final, color="tab:green", ls="--", lw=1, alpha=0.8)
fig_p.suptitle("Pulse-time shape", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Common plus-preparation system.  All benchmark backends below use this same nn Hamiltonian.
Lx, Ly = 3, 3
spacing = 10.0
N = Lx * Ly
interaction_mode = "nn"

prep_steps = 200
prep_proto = segments_protocol([full_segments[0]], n_steps=prep_steps)
geom = Register.rectangle(Lx, Ly, spacing_um=spacing)
prep_system = RydbergSystem.from_lattice(
    geom,
    level_structure="01r",
    interaction=InteractionSpec(C6=DEFAULT_C6, mode=interaction_mode),
    protocol=prep_proto,
    Omega=OMEGA_R,
)
t_eval_prep = np.linspace(0.0, t_pi2_R, 101)

# Exact target in the full 01r Hilbert space: product_i (|1> - i|r>) / sqrt(2).
plus_target_exact = sum(
    (-1j) ** sum(c == "r" for c in cfg) * prep_system.product_state(list(cfg))
    for cfg in product(["1", "r"], repeat=N)
) / (2 ** (N / 2))

# TeNPy/MPS target for TN fidelity checks.
prep_spec = tn_lattice_spec_from_system(prep_system)
plus_target_mps = product_superposition_mps(
    prep_spec,
    zero_amp=0.0,
    ground_amp=1 / np.sqrt(2),
    rydberg_amp=-1j / np.sqrt(2),
)

plus_results = {}
print(f"Plus-prep benchmark: {Lx}x{Ly}, level_structure=01r, interaction={interaction_mode}, steps={prep_steps}")
print(f"t_pi2_R = {t_pi2_R:.3e} s")


## Exact sparse-expm baseline

This block runs exact state-vector evolution with the same nearest-neighbor Hamiltonian. It is the reference for observables and target-state fidelity.


In [ ]:
# Exact sparse-expm plus-state preparation.
backend_name = "sparse_expm"
_t0 = time.perf_counter()
res_prep_exact = simulate(
    prep_system,
    [],
    "all_1",
    backend=backend_name,
    backend_options={"n_steps": prep_proto.n_steps},
    t_eval=t_eval_prep,
)
elapsed = time.perf_counter() - _t0

psi0 = prep_system.product_state(["1"] * N)
t_prep = np.concatenate([[0.0], res_prep_exact.times])
states = [psi0, *res_prep_exact.states]
n_0_prep = np.array([[prep_system.expectation(f"n_0_{i}", psi) for i in range(N)] for psi in states])
n_1_prep = np.array([[prep_system.expectation(f"n_1_{i}", psi) for i in range(N)] for psi in states])
n_r_prep = np.array([[prep_system.expectation(f"n_r_{i}", psi) for i in range(N)] for psi in states])
psi_plus_exact = res_prep_exact.psi_final
fid_prep = float(np.abs(np.vdot(plus_target_exact, psi_plus_exact)) ** 2)

plus_results[backend_name] = {
    "status": "ok",
    "elapsed_s": elapsed,
    "fidelity": fid_prep,
    "times": t_prep,
    "n_0": n_0_prep,
    "n_1": n_1_prep,
    "n_r": n_r_prep,
    "final_n_r_mean": float(n_r_prep[-1].mean()),
    "max_mean_diff_vs_exact": 0.0,
    "max_site_diff_vs_exact": 0.0,
}

print(f"{backend_name}: time={elapsed:.3f} s, F={fid_prep:.6f}, final mean <n_r>={n_r_prep[-1].mean():.6f}")

BS = chr(92)
colors = {"0": "tab:gray", "1": "tab:blue", "r": "tab:red"}
fig, axes = plt.subplots(Ly, Lx, figsize=(3.5 * Lx, 3 * Ly), sharex=True, sharey=True)
axes = np.asarray(axes).reshape(Ly, Lx)
for i in range(N):
    ix, iy = i // Ly, i % Ly
    ax = axes[iy, ix]
    ax.plot(t_prep, n_0_prep[:, i], color=colors["0"], lw=2, label=f"${BS}langle n_0{BS}rangle$")
    ax.plot(t_prep, n_1_prep[:, i], color=colors["1"], lw=2, label=f"${BS}langle n_1{BS}rangle$")
    ax.plot(t_prep, n_r_prep[:, i], color=colors["r"], lw=2, label=f"${BS}langle n_r{BS}rangle$")
    ax.axhline(0.5, color="k", ls=":", lw=1)
    ax.set_title(f"({ix}, {iy})")
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(loc="upper right", ncol=3)
for ax in axes[-1, :]:
    ax.set_xlabel("time (s)")
fig.suptitle(f"{Lx}x{Ly} plus-state preparation: local populations ({backend_name}, {interaction_mode})")
plt.tight_layout()
plt.show()


## TeNPy MPS-TDVP (`mps`)


In [ ]:
# TeNPy MPS-TDVP plus-state preparation.
backend_name = "mps"
backend_opts = {
    "chi_max": 128,
    "dt": t_pi2_R / prep_proto.n_steps,
    "svd_min": 1e-10,
}
_t0 = time.perf_counter()
res_prep_tenpy = simulate(
    prep_system,
    [],
    "all_1",
    backend=backend_name,
    backend_options=backend_opts,
    method="tdvp",
    t_eval=t_eval_prep,
    observables=["n_0", "n_1", "n_i", "n_mean"],
)
elapsed = time.perf_counter() - _t0

t_prep = np.asarray(res_prep_tenpy.times, dtype=float)
n_0_prep = np.asarray(res_prep_tenpy.metadata["obs"]["n_0"], dtype=float)
n_1_prep = np.asarray(res_prep_tenpy.metadata["obs"]["n_1"], dtype=float)
n_r_prep = np.asarray(res_prep_tenpy.metadata["obs"]["n_i"], dtype=float)
psi_plus_tenpy = res_prep_tenpy.psi_final
fid_prep = mps_fidelity(plus_target_mps, psi_plus_tenpy)

max_mean_diff = np.nan
max_site_diff = np.nan
if "sparse_expm" in plus_results and plus_results["sparse_expm"]["status"] == "ok":
    ref_t = plus_results["sparse_expm"]["times"]
    ref_n = plus_results["sparse_expm"]["n_r"]
    ref_match = np.asarray([ref_n[int(np.argmin(np.abs(ref_t - t)))] for t in t_prep])
    max_mean_diff = float(np.max(np.abs(n_r_prep.mean(axis=1) - ref_match.mean(axis=1))))
    max_site_diff = float(np.max(np.abs(n_r_prep - ref_match)))

plus_results[backend_name] = {
    "status": "ok",
    "elapsed_s": elapsed,
    "fidelity": fid_prep,
    "times": t_prep,
    "n_0": n_0_prep,
    "n_1": n_1_prep,
    "n_r": n_r_prep,
    "final_n_r_mean": float(n_r_prep[-1].mean()),
    "max_mean_diff_vs_exact": max_mean_diff,
    "max_site_diff_vs_exact": max_site_diff,
}

print(f"{backend_name}: time={elapsed:.3f} s, F={fid_prep:.6f}, final mean <n_r>={n_r_prep[-1].mean():.6f}")
print(f"max |mean <n_r> - exact|={max_mean_diff:.3e}, max site diff={max_site_diff:.3e}")

BS = chr(92)
colors = {"0": "tab:gray", "1": "tab:blue", "r": "tab:red"}
fig, axes = plt.subplots(Ly, Lx, figsize=(3.5 * Lx, 3 * Ly), sharex=True, sharey=True)
axes = np.asarray(axes).reshape(Ly, Lx)
for i in range(N):
    ix, iy = i // Ly, i % Ly
    ax = axes[iy, ix]
    ax.plot(t_prep, n_0_prep[:, i], color=colors["0"], lw=2, label=f"${BS}langle n_0{BS}rangle$")
    ax.plot(t_prep, n_1_prep[:, i], color=colors["1"], lw=2, label=f"${BS}langle n_1{BS}rangle$")
    ax.plot(t_prep, n_r_prep[:, i], color=colors["r"], lw=2, label=f"${BS}langle n_r{BS}rangle$")
    ax.axhline(0.5, color="k", ls=":", lw=1)
    ax.set_title(f"({ix}, {iy})")
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(loc="upper right", ncol=3)
for ax in axes[-1, :]:
    ax.set_xlabel("time (s)")
fig.suptitle(f"{Lx}x{Ly} plus-state preparation: local populations ({backend_name}, {interaction_mode})")
plt.tight_layout()
plt.show()


## YASTN finite PEPS (`peps`)

This block runs the same nearest-neighbor `01r` problem with the YASTN finite-PEPS backend. PEPS keeps the 2D lattice geometry and uses local dimension `d=3`; this block reports observable consistency against exact rather than target-state fidelity.


In [ ]:
# YASTN finite-PEPS plus-state preparation.
backend_name = "peps"
backend_opts = {
    "chi_max": 8,
    "dt": t_pi2_R / prep_proto.n_steps,
    "svd_min": 1e-10,
    "update_environment": "ntu",
    "measurement_environment": "bp",
    "use_cuda": False,
}
_t0 = time.perf_counter()
try:
    res_prep_peps = simulate(
        prep_system,
        [],
        "all_1",
        backend=backend_name,
        backend_options=backend_opts,
        method="tdvp",
        t_eval=t_eval_prep,
        observables=["n_0", "n_1", "n_r", "n_mean"],
    )
    elapsed = time.perf_counter() - _t0

    t_prep = np.asarray(res_prep_peps.times, dtype=float)
    n_0_prep = np.asarray(res_prep_peps.metadata["obs"]["n_0"], dtype=float)
    n_1_prep = np.asarray(res_prep_peps.metadata["obs"]["n_1"], dtype=float)
    n_r_prep = np.asarray(res_prep_peps.metadata["obs"]["n_r"], dtype=float)

    max_mean_diff = np.nan
    max_site_diff = np.nan
    if "sparse_expm" in plus_results and plus_results["sparse_expm"]["status"] == "ok":
        ref_t = plus_results["sparse_expm"]["times"]
        ref_n = plus_results["sparse_expm"]["n_r"]
        ref_match = np.asarray([ref_n[int(np.argmin(np.abs(ref_t - t)))] for t in t_prep])
        max_mean_diff = float(np.max(np.abs(n_r_prep.mean(axis=1) - ref_match.mean(axis=1))))
        max_site_diff = float(np.max(np.abs(n_r_prep - ref_match)))

    plus_results[backend_name] = {
        "status": "ok",
        "elapsed_s": elapsed,
        "fidelity": np.nan,
        "times": t_prep,
        "n_0": n_0_prep,
        "n_1": n_1_prep,
        "n_r": n_r_prep,
        "final_n_r_mean": float(n_r_prep[-1].mean()),
        "max_mean_diff_vs_exact": max_mean_diff,
        "max_site_diff_vs_exact": max_site_diff,
    }

    print(f"{backend_name}: time={elapsed:.3f} s, final mean <n_r>={n_r_prep[-1].mean():.6f}")
    print(f"max |mean <n_r> - exact|={max_mean_diff:.3e}, max site diff={max_site_diff:.3e}")

    BS = chr(92)
    colors = {"0": "tab:gray", "1": "tab:blue", "r": "tab:red"}
    fig, axes = plt.subplots(Ly, Lx, figsize=(3.5 * Lx, 3 * Ly), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(Ly, Lx)
    for i in range(N):
        ix, iy = i // Ly, i % Ly
        ax = axes[iy, ix]
        ax.plot(t_prep, n_0_prep[:, i], color=colors["0"], lw=2, label=f"${BS}langle n_0{BS}rangle$")
        ax.plot(t_prep, n_1_prep[:, i], color=colors["1"], lw=2, label=f"${BS}langle n_1{BS}rangle$")
        ax.plot(t_prep, n_r_prep[:, i], color=colors["r"], lw=2, label=f"${BS}langle n_r{BS}rangle$")
        ax.axhline(0.5, color="k", ls=":", lw=1)
        ax.set_title(f"({ix}, {iy})")
        ax.grid(alpha=0.3)
        if i == 0:
            ax.legend(loc="upper right", ncol=3)
    for ax in axes[-1, :]:
        ax.set_xlabel("time (s)")
    fig.suptitle(f"{Lx}x{Ly} plus-state preparation: local populations ({backend_name}, {interaction_mode})")
    plt.tight_layout()
    plt.show()

except Exception as exc:
    elapsed = time.perf_counter() - _t0
    plus_results[backend_name] = {
        "status": "failed",
        "elapsed_s": elapsed,
        "error": repr(exc),
        "fidelity": np.nan,
        "final_n_r_mean": np.nan,
        "max_mean_diff_vs_exact": np.nan,
        "max_site_diff_vs_exact": np.nan,
    }
    print(f"{backend_name}: failed after {elapsed:.3f} s")
    print(repr(exc))


## CUDA GPUTN

This block runs the CUDA GPUTN path for the same nearest-neighbor `01r` problem. If CUDA/cuQuantum dependencies are missing, the block prints the dependency error and records that status without changing the other results.


In [ ]:
# CUDA GPUTN plus-state preparation.
backend_name = "gputn"
backend_opts = {
    "chi_max": 128,
    "dt": t_pi2_R / prep_proto.n_steps,
    "svd_min": 1e-10,
    "device_id": 0,
    "require_gpu": True,
    "kernel": "statevector",
}

_t0 = time.perf_counter()
try:
    res_prep_gputn = simulate(
        prep_system,
        [],
        "all_1",
        backend=backend_name,
        backend_options=backend_opts,
        method="tdvp",
        t_eval=t_eval_prep,
        observables=["n_0", "n_1", "n_i", "n_mean"],
    )
    elapsed = time.perf_counter() - _t0

    t_prep = np.asarray(res_prep_gputn.times, dtype=float)
    n_0_prep = np.asarray(res_prep_gputn.metadata["obs"]["n_0"], dtype=float)
    n_1_prep = np.asarray(res_prep_gputn.metadata["obs"]["n_1"], dtype=float)
    n_r_prep = np.asarray(res_prep_gputn.metadata["obs"]["n_i"], dtype=float)

    max_mean_diff = np.nan
    max_site_diff = np.nan
    if "sparse_expm" in plus_results and plus_results["sparse_expm"]["status"] == "ok":
        ref_t = plus_results["sparse_expm"]["times"]
        ref_n = plus_results["sparse_expm"]["n_r"]
        ref_match = np.asarray([ref_n[int(np.argmin(np.abs(ref_t - t)))] for t in t_prep])
        max_mean_diff = float(np.max(np.abs(n_r_prep.mean(axis=1) - ref_match.mean(axis=1))))
        max_site_diff = float(np.max(np.abs(n_r_prep - ref_match)))

    plus_results[backend_name] = {
        "status": "ok",
        "elapsed_s": elapsed,
        "fidelity": np.nan,
        "times": t_prep,
        "n_0": n_0_prep,
        "n_1": n_1_prep,
        "n_r": n_r_prep,
        "final_n_r_mean": float(n_r_prep[-1].mean()),
        "max_mean_diff_vs_exact": max_mean_diff,
        "max_site_diff_vs_exact": max_site_diff,
    }

    print(f"{backend_name}: time={elapsed:.3f} s, final mean <n_r>={n_r_prep[-1].mean():.6f}")
    print(f"max |mean <n_r> - exact|={max_mean_diff:.3e}, max site diff={max_site_diff:.3e}")

    BS = chr(92)
    colors = {"0": "tab:gray", "1": "tab:blue", "r": "tab:red"}
    fig, axes = plt.subplots(Ly, Lx, figsize=(3.5 * Lx, 3 * Ly), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(Ly, Lx)
    for i in range(N):
        ix, iy = i // Ly, i % Ly
        ax = axes[iy, ix]
        ax.plot(t_prep, n_0_prep[:, i], color=colors["0"], lw=2, label=f"${BS}langle n_0{BS}rangle$")
        ax.plot(t_prep, n_1_prep[:, i], color=colors["1"], lw=2, label=f"${BS}langle n_1{BS}rangle$")
        ax.plot(t_prep, n_r_prep[:, i], color=colors["r"], lw=2, label=f"${BS}langle n_r{BS}rangle$")
        ax.axhline(0.5, color="k", ls=":", lw=1)
        ax.set_title(f"({ix}, {iy})")
        ax.grid(alpha=0.3)
        if i == 0:
            ax.legend(loc="upper right", ncol=3)
    for ax in axes[-1, :]:
        ax.set_xlabel("time (s)")
    fig.suptitle(f"{Lx}x{Ly} plus-state preparation: local populations ({backend_name}, {interaction_mode})")
    plt.tight_layout()
    plt.show()

except Exception as exc:
    elapsed = time.perf_counter() - _t0
    plus_results[backend_name] = {
        "status": "failed",
        "elapsed_s": elapsed,
        "error": repr(exc),
        "fidelity": np.nan,
        "final_n_r_mean": np.nan,
        "max_mean_diff_vs_exact": np.nan,
        "max_site_diff_vs_exact": np.nan,
    }
    print(f"{backend_name}: failed after {elapsed:.3f} s")
    print(repr(exc))


## Backend comparison

Run this block after the backend blocks above. It prints the timing/error list and plots runtime plus observable consistency. The fastest successful method is the one with the smallest wall time in this table.


In [ ]:
# Compare backend timing and consistency.
print("backend        status       time(s)    fidelity    final <n_r>    max mean diff    max site diff")
for name, row in plus_results.items():
    elapsed = row.get("elapsed_s", np.nan)
    fid = row.get("fidelity", np.nan)
    final_mean = row.get("final_n_r_mean", np.nan)
    mean_diff = row.get("max_mean_diff_vs_exact", np.nan)
    site_diff = row.get("max_site_diff_vs_exact", np.nan)
    print(
        f"{name:<14} {row.get('status', ''):<10} "
        f"{elapsed:9.3f}  {fid:10.6f}  {final_mean:12.6f}  "
        f"{mean_diff:13.3e}  {site_diff:13.3e}"
    )

ok_names = [name for name, row in plus_results.items() if row.get("status") == "ok"]
if ok_names:
    fastest = min(ok_names, key=lambda name: plus_results[name]["elapsed_s"])
    print(f"\nFastest successful backend: {fastest}")

    fig_cmp, axes_cmp = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    axes_cmp[0].bar(ok_names, [plus_results[name]["elapsed_s"] for name in ok_names], color="tab:blue", alpha=0.8)
    axes_cmp[0].set_ylabel("wall time (s)")
    axes_cmp[0].set_title("Runtime")
    axes_cmp[0].tick_params(axis="x", rotation=25)

    for name in ok_names:
        row = plus_results[name]
        axes_cmp[1].plot(row["times"], row["n_r"].mean(axis=1), marker="o", ms=3, lw=1.5, label=name)
    axes_cmp[1].set_xlabel("time (s)")
    axes_cmp[1].set_ylabel(r"mean $\langle n_r \rangle$")
    axes_cmp[1].set_title("Mean Rydberg population")
    axes_cmp[1].grid(alpha=0.3)
    axes_cmp[1].legend(fontsize=8)

    axes_cmp[2].bar(
        ok_names,
        [plus_results[name].get("max_site_diff_vs_exact", np.nan) for name in ok_names],
        color="tab:orange",
        alpha=0.8,
    )
    axes_cmp[2].set_ylabel("max site diff vs exact")
    axes_cmp[2].set_title("Observable consistency")
    axes_cmp[2].set_yscale("symlog", linthresh=1e-10)
    axes_cmp[2].tick_params(axis="x", rotation=25)
    plt.show()

print("\nInterpretation:")
print("- All successful TN blocks use the same nn Hamiltonian as exact, so differences are TDVP/Trotter/truncation errors rather than interaction-graph mismatch.")
print("- MPS uses a chain ordering; PEPS keeps the 2D geometry. Their cost/accuracy depends strongly on chi_max, dt, and the PEPS environment choice.")
print("- GPUTN is expected to be fastest only when CUDA/cuQuantum are available and the problem is large enough to amortize GPU overhead.")


## Sublattice-X echo strategies

The following blocks keep the notebook examples from the original script, but use `interaction_mode="nn"` explicitly. They are exact-state-vector runs so each block directly produces its result and plot.


In [ ]:
# Delta echo: build schedule, run exact, and plot fidelity/population.
echo_M = 2
echo_width = t_pi2_R / 80
off_detuning = 20 * OMEGA_R
echo_interaction_mode = interaction_mode
V_nn_echo = DEFAULT_C6 / spacing**6

mask_A = np.array([((ix + iy) % 2 == 0) for ix in range(Lx) for iy in range(Ly)], dtype=bool)
mask_B = ~mask_A

if echo_M < 0:
    raise ValueError("echo_M must be nonnegative")
if echo_M * echo_width >= t_pi2_R:
    raise ValueError("Echo pulse width is too large for the preparation window.")

if echo_M == 0:
    delta_echo_segments = [Segment(duration=t_pi2_R, omega_R=OMEGA_R)]
else:
    delta_echo_interval = (t_pi2_R - echo_M * echo_width) / (echo_M + 1)
    delta_echo_x_omega = np.pi / echo_width
    delta_echo_segments = []
    for _ in range(echo_M):
        delta_echo_segments.append(Segment(duration=delta_echo_interval, omega_R=OMEGA_R))
        delta_echo_segments.append(
            Segment(
                duration=echo_width,
                omega_R=delta_echo_x_omega,
                delta_R=np.where(mask_A, 0.0, off_detuning),
            )
        )
    delta_echo_segments.append(Segment(duration=delta_echo_interval, omega_R=OMEGA_R))

delta_echo_proto = segments_protocol(delta_echo_segments, n_steps=max(24, 20 * len(delta_echo_segments)))

delta_echo_edges = [0.0]
for seg in delta_echo_proto.segments:
    delta_echo_edges.append(delta_echo_edges[-1] + seg.duration)

fig_de, axes_de = plt.subplots(4, 1, figsize=(12, 7), sharex=True)
for ax_de, (field, label) in zip(axes_de, _proto_fields):
    plotted_ab = False
    for j, seg in enumerate(delta_echo_proto.segments):
        prof = np.asarray(getattr(seg, field), dtype=float)
        if prof.ndim == 0:
            prof = np.full(N, float(prof))
        t0, t1 = delta_echo_edges[j], delta_echo_edges[j + 1]
        if np.allclose(prof, prof[0]):
            ax_de.hlines(prof[0], t0, t1, colors="tab:blue", lw=2)
        else:
            ax_de.hlines(prof[mask_A].mean(), t0, t1, colors="tab:red", lw=2)
            ax_de.hlines(prof[mask_B].mean(), t0, t1, colors="tab:green", lw=2, ls="--")
            plotted_ab = True
    if plotted_ab:
        ax_de.plot([], [], color="tab:red", lw=2, label="A sublattice")
        ax_de.plot([], [], color="tab:green", lw=2, ls="--", label="B sublattice")
        ax_de.legend(loc="upper right", fontsize=8)
    ax_de.axhline(0.0, color="k", ls=":", lw=0.8)
    ax_de.set_ylabel(label)
    ax_de.grid(alpha=0.3)
    ax_de.set_xlim(0.0, delta_echo_edges[-1])
axes_de[-1].set_xlabel("time (s)")
fig_de.suptitle("Pulse schedule: sublattice-X delta echo", y=1.01)
plt.tight_layout()
plt.show()

delta_system = RydbergSystem.from_lattice(
    geom,
    level_structure="01r",
    interaction=InteractionSpec(C6=DEFAULT_C6, mode=echo_interaction_mode),
    protocol=delta_echo_proto,
    Omega=OMEGA_R,
)
t_eval_delta = np.linspace(0.0, delta_echo_proto._t_gate, 25)
_t0 = time.perf_counter()
res_delta = simulate(delta_system, [], "all_1", backend="sparse_expm", backend_options={"n_steps": delta_echo_proto.n_steps}, t_eval=t_eval_delta)
elapsed_delta = time.perf_counter() - _t0

target_delta = plus_target_exact
fid_delta = float(np.abs(np.vdot(target_delta, res_delta.psi_final)) ** 2)
psi0_delta = delta_system.product_state(["1"] * N)
t_delta = np.concatenate([[0.0], res_delta.times])
states_delta = [psi0_delta, *res_delta.states]
n_r_delta_mean = np.array([
    np.mean([delta_system.expectation(f"n_r_{i}", psi) for i in range(N)])
    for psi in states_delta
])

print(f"delta echo exact ({echo_interaction_mode}): time={elapsed_delta:.3f} s, F={fid_delta:.6f}, final mean <n_r>={n_r_delta_mean[-1]:.6f}")

fig_delta, ax_delta = plt.subplots(figsize=(7, 4))
ax_delta.plot(t_delta, n_r_delta_mean, lw=2, color="tab:purple")
ax_delta.set_xlabel("time (s)")
ax_delta.set_ylabel("mean Rydberg population")
ax_delta.set_title(f"Delta echo population ({Lx}x{Ly}, 01r, {echo_interaction_mode})")
ax_delta.grid(alpha=0.3)
plt.tight_layout()
plt.show()

echo_comparison = [{
    "strategy": "sublattice-X delta echo",
    "fidelity": fid_delta,
    "times": t_delta,
    "n_r_mean": n_r_delta_mean,
    "final_n_r_mean": float(n_r_delta_mean[-1]),
    "elapsed_s": elapsed_delta,
}]


In [ ]:
# Ordered echo: build schedule, run exact, and compare with delta echo if available.
ordered_z_max = 4
ordered_echo_segments = [
    Segment(
        duration=t_pi2_R,
        omega_R=OMEGA_R,
        delta_R=np.where(mask_A, 0.0, off_detuning),
    ),
]
for k in range(ordered_z_max + 1):
    ordered_echo_segments.append(
        Segment(
            duration=t_pi2_R,
            omega_R=OMEGA_R,
            delta_R=np.where(mask_B, k * V_nn_echo, off_detuning),
        )
    )

ordered_echo_proto = segments_protocol(ordered_echo_segments, n_steps=max(24, 20 * len(ordered_echo_segments)))
ordered_echo_edges = [0.0]
for seg in ordered_echo_proto.segments:
    ordered_echo_edges.append(ordered_echo_edges[-1] + seg.duration)

fig_oe, axes_oe = plt.subplots(4, 1, figsize=(12, 7), sharex=True)
for ax_oe, (field, label) in zip(axes_oe, _proto_fields):
    plotted_ab = False
    for j, seg in enumerate(ordered_echo_proto.segments):
        prof = np.asarray(getattr(seg, field), dtype=float)
        if prof.ndim == 0:
            prof = np.full(N, float(prof))
        t0, t1 = ordered_echo_edges[j], ordered_echo_edges[j + 1]
        if np.allclose(prof, prof[0]):
            ax_oe.hlines(prof[0], t0, t1, colors="tab:blue", lw=2)
        else:
            ax_oe.hlines(prof[mask_A].mean(), t0, t1, colors="tab:red", lw=2)
            ax_oe.hlines(prof[mask_B].mean(), t0, t1, colors="tab:green", lw=2, ls="--")
            plotted_ab = True
    if plotted_ab:
        ax_oe.plot([], [], color="tab:red", lw=2, label="A sublattice")
        ax_oe.plot([], [], color="tab:green", lw=2, ls="--", label="B sublattice")
        ax_oe.legend(loc="upper right", fontsize=8)
    ax_oe.axhline(0.0, color="k", ls=":", lw=0.8)
    ax_oe.set_ylabel(label)
    ax_oe.grid(alpha=0.3)
    ax_oe.set_xlim(0.0, ordered_echo_edges[-1])
axes_oe[-1].set_xlabel("time (s)")
fig_oe.suptitle("Pulse schedule: sublattice-X ordered echo", y=1.01)
plt.tight_layout()
plt.show()

ordered_system = RydbergSystem.from_lattice(
    geom,
    level_structure="01r",
    interaction=InteractionSpec(C6=DEFAULT_C6, mode=echo_interaction_mode),
    protocol=ordered_echo_proto,
    Omega=OMEGA_R,
)
t_eval_ordered = np.linspace(0.0, ordered_echo_proto._t_gate, 25)
_t0 = time.perf_counter()
res_ordered = simulate(ordered_system, [], "all_1", backend="sparse_expm", backend_options={"n_steps": ordered_echo_proto.n_steps}, t_eval=t_eval_ordered)
elapsed_ordered = time.perf_counter() - _t0

fid_ordered = float(np.abs(np.vdot(plus_target_exact, res_ordered.psi_final)) ** 2)
psi0_ordered = ordered_system.product_state(["1"] * N)
t_ordered = np.concatenate([[0.0], res_ordered.times])
states_ordered = [psi0_ordered, *res_ordered.states]
n_r_ordered_mean = np.array([
    np.mean([ordered_system.expectation(f"n_r_{i}", psi) for i in range(N)])
    for psi in states_ordered
])

ordered_echo_row = {
    "strategy": "sublattice-X ordered echo",
    "fidelity": fid_ordered,
    "times": t_ordered,
    "n_r_mean": n_r_ordered_mean,
    "final_n_r_mean": float(n_r_ordered_mean[-1]),
    "elapsed_s": elapsed_ordered,
}
echo_comparison = [row for row in globals().get("echo_comparison", []) if row["strategy"] != ordered_echo_row["strategy"]]
echo_comparison.append(ordered_echo_row)

print(f"ordered echo exact ({echo_interaction_mode}): time={elapsed_ordered:.3f} s, F={fid_ordered:.6f}, final mean <n_r>={n_r_ordered_mean[-1]:.6f}")
print(f"Best fidelity: {max(echo_comparison, key=lambda x: x['fidelity'])['strategy']}")

fig_echo, axes_echo = plt.subplots(1, 2, figsize=(12, 4))
labels = [row["strategy"] for row in echo_comparison]
fids = [row["fidelity"] for row in echo_comparison]
axes_echo[0].bar(labels, fids, color=["tab:purple", "tab:brown"][:len(echo_comparison)])
axes_echo[0].set_ylim(max(0.0, min(fids) - 0.05), 1.005)
axes_echo[0].set_ylabel("target fidelity")
axes_echo[0].tick_params(axis="x", rotation=15)
axes_echo[0].grid(axis="y", alpha=0.3)
for idx, fid in enumerate(fids):
    axes_echo[0].text(idx, fid, f"{fid:.3f}", ha="center", va="bottom", fontsize=9)

for row in echo_comparison:
    axes_echo[1].plot(row["times"], row["n_r_mean"], lw=2, label=row["strategy"])
axes_echo[1].set_xlabel("time (s)")
axes_echo[1].set_ylabel("mean Rydberg population")
axes_echo[1].grid(alpha=0.3)
axes_echo[1].legend(fontsize=8)
fig_echo.suptitle(f"Sublattice-X echo benchmark ({Lx}x{Ly}, 01r, {echo_interaction_mode})")
plt.tight_layout()
plt.show()


## Exact Ising evolution after preparation

This block uses the exact plus-preparation final state as the initial condition for the Ising segment, again with `mode="nn"`.


In [ ]:
# Stage 2: exact Ising evolution using the exact prep final state.
ising_proto = segments_protocol([full_segments[1]], n_steps=200)
ising_system = prep_system.with_protocol(ising_proto)
t_eval_ising = np.linspace(0.0, T_ISING, 101)

_t0 = time.perf_counter()
res_ising = simulate(
    ising_system,
    [],
    psi_plus_exact,
    backend="sparse_expm",
    backend_options={"n_steps": ising_proto.n_steps},
    t_eval=t_eval_ising,
)
elapsed_ising = time.perf_counter() - _t0

t_ising = np.concatenate([[0.0], res_ising.times])
states_ising = [psi_plus_exact, *res_ising.states]
n_r_ising = np.array([[ising_system.expectation(f"n_r_{i}", psi) for i in range(N)] for psi in states_ising])

t_offset = plus_results["sparse_expm"]["times"][-1]
t_full = np.concatenate([plus_results["sparse_expm"]["times"], t_offset + t_ising[1:]])
n_r_full = np.vstack([plus_results["sparse_expm"]["n_r"], n_r_ising[1:]])

fig, axes = plt.subplots(Ly, Lx, figsize=(3.5 * Lx, 3 * Ly), sharex=True, sharey=True)
axes = np.asarray(axes).reshape(Ly, Lx)
for i in range(N):
    ix, iy = i // Ly, i % Ly
    ax = axes[iy, ix]
    ax.plot(t_full, n_r_full[:, i], color="tab:red", lw=2, label=r"$\langle n_r\rangle$")
    ax.axvline(t_prep_end, color="tab:orange", ls="--", lw=1, alpha=0.8)
    ax.set_title(f"({ix}, {iy})")
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(loc="upper right")
for ax in axes[-1, :]:
    ax.set_xlabel("time (s)")
fig.suptitle(f"Full process local Rydberg density: prep + Ising evolution ({Lx}x{Ly}, {interaction_mode})")
plt.tight_layout()
plt.show()

final_n_r = n_r_full[-1]
print(f"Ising exact ({interaction_mode}): time={elapsed_ising:.3f} s")
print("Final local Rydberg density <n_r_i>(t_final):")
print(f"  mean={final_n_r.mean():.6f}, min={final_n_r.min():.6f}, max={final_n_r.max():.6f}")
for i in range(N):
    print(f"  site {i}: {final_n_r[i]:.6f}")
